In [15]:
# instal libs

!pip install langchain langchain-community langchain-huggingface
!pip install faiss-cpu
!pip install pypdf
!pip install sentence-transformers
!pip install groq langchain-groq
!pip install langchain-text-splitters
!pip install streamlit pyngrok

print("ok 1")

ok 1


In [16]:
import os
import glob

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq



print("ok 2")

ok 2


In [17]:
# download 7 papers to collab


import os
import urllib.request

os.makedirs('papers', exist_ok=True)

papers = {
    "attention_is_all_you_need.pdf": "https://arxiv.org/pdf/1706.03762.pdf",
    "bert.pdf": "https://arxiv.org/pdf/1810.04805.pdf",
    "gpt3.pdf": "https://arxiv.org/pdf/2005.14165.pdf",
    "rag.pdf": "https://arxiv.org/pdf/2005.11401.pdf",
    "sentence_bert.pdf": "https://arxiv.org/pdf/1908.10084.pdf",
    "lora.pdf": "https://arxiv.org/pdf/2106.09685.pdf",
    "llama2.pdf": "https://arxiv.org/pdf/2307.09288.pdf"
}

for filename, url in papers.items():
    filepath = f"papers/{filename}"
    if os.path.exists(filepath):
        print(f"Already exists: {filename}")
    else:
        print(f"Downloading: {filename}...")
        urllib.request.urlretrieve(url, filepath)
        print(f"Done: {filename}")

print(f"\nAll papers downloaded!")
print(f"Files in papers/: {os.listdir('papers')}")

print("ok 3")

Already exists: attention_is_all_you_need.pdf
Already exists: bert.pdf
Already exists: gpt3.pdf
Already exists: rag.pdf
Already exists: sentence_bert.pdf
Already exists: lora.pdf
Already exists: llama2.pdf

All papers downloaded!
Files in papers/: ['sentence_bert.pdf', 'bert.pdf', 'lora.pdf', 'attention_is_all_you_need.pdf', 'rag.pdf', 'gpt3.pdf', 'llama2.pdf']
ok 3


In [18]:
# load the content

from langchain_community.document_loaders import PyPDFLoader
import glob

pdf_files = glob.glob("papers/*.pdf")
print(f"Found {len(pdf_files)} PDF files:")
for f in pdf_files:
    print(f"  - {f}")

all_documents = []

for pdf_path in pdf_files:
    print(f"\nLoading: {pdf_path}")

    loader = PyPDFLoader(pdf_path)


    pages = loader.load()
    all_documents.extend(pages)

    print(f"  Loaded {len(pages)} pages")
    print(f"  Preview: {pages[0].page_content[:200]}...")

print(f"\nTotal pages across all papers: {len(all_documents)}")
print("\nSample document metadata:", all_documents[0].metadata)

print("ok 4")

Found 7 PDF files:
  - papers/sentence_bert.pdf
  - papers/bert.pdf
  - papers/lora.pdf
  - papers/attention_is_all_you_need.pdf
  - papers/rag.pdf
  - papers/gpt3.pdf
  - papers/llama2.pdf

Loading: papers/sentence_bert.pdf
  Loaded 11 pages
  Preview: Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks
Nils Reimers and Iryna Gurevych
Ubiquitous Knowledge Processing Lab (UKP-TUDA)
Department of Computer Science, Technische Universit¨at Da...

Loading: papers/bert.pdf
  Loaded 16 pages
  Preview: BERT: Pre-training of Deep Bidirectional Transformers for
Language Understanding
Jacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova
Google AI Language
{jacobdevlin,mingweichang,kentonl,kristout...

Loading: papers/lora.pdf
  Loaded 26 pages
  Preview: LORA: L OW-R ANK ADAPTATION OF LARGE LAN-
GUAGE MODELS
Edward Hu∗ Yelong Shen∗ Phillip Wallis Zeyuan Allen-Zhu
Yuanzhi Li Shean Wang Lu Wang Weizhu Chen
Microsoft Corporation
{edwardhu, yeshe, phwalli...

Loading: papers/attenti

In [19]:
# spliting the document into chunks

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
     separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_documents(all_documents)
print(f"Original pages: {len(all_documents)}")
print(f"Chunks after splitting: {len(chunks)}")
print(f"Average chunks per page: {len(chunks)/len(all_documents):.1f}")
print(f"\nSample chunk content:")
print(f"  Text: {chunks[5].page_content[:300]}...")
print(f"  Metadata: {chunks[5].metadata}")
print(f"  Length: {len(chunks[5].page_content)} characters")

print("ok 5")

Original pages: 239
Chunks after splitting: 1057
Average chunks per page: 4.4

Sample chunk content:
  Text: most similar sentence pair in a collection of 10,000
sentences is reduced from 65 hours with BERT to
the computation of 10,000 sentence embeddings
(~5 seconds with SBERT) and computing cosine-
similarity (~0.01 seconds). By using optimized
index structures, ﬁnding the most similar Quora
question can...
  Metadata: {'producer': 'pdfTeX-1.40.17', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-08-28T00:45:27+00:00', 'author': '', 'keywords': '', 'moddate': '2019-08-28T00:45:27+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.17 (TeX Live 2016) kpathsea version 6.2.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'papers/sentence_bert.pdf', 'total_pages': 11, 'page': 1, 'page_label': '2'}
  Length: 974 characters
ok 5


In [20]:
# creating embedding and build vector
print("Loading embedding model...")
print("(This downloads ~90MB model on first run — takes 1-2 minutes)")

embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
   )

print("Embedding model loaded!")
print("\nCreating vector database from chunks...")
print("(Converting each chunk to a 384-dimensional vector)")
print("(This takes 2-5 minutes depending on number of chunks)")

vectorstore = FAISS.from_documents(
    chunks,
    embedding_model
)

vectorstore.save_local("faiss_index")
print(f"\nVector database created with {len(chunks)} vectors!")
print("Saved to faiss_index/")
print("Each vector has 384 dimensions")


test_query = "What is attention mechanism?"
results = vectorstore.similarity_search(test_query, k=2)
print(f"\nTest search: '{test_query}'")
print(f"Top result preview: {results[0].page_content[:200]}...")
print(f"From paper: {results[0].metadata.get('source', 'unknown')}")

print("ok 6")

Loading embedding model...
(This downloads ~90MB model on first run — takes 1-2 minutes)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!

Creating vector database from chunks...
(Converting each chunk to a 384-dimensional vector)
(This takes 2-5 minutes depending on number of chunks)

Vector database created with 1057 vectors!
Saved to faiss_index/
Each vector has 384 dimensions

Test search: 'What is attention mechanism?'
Top result preview: 3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as ...
From paper: papers/attention_is_all_you_need.pdf
ok 6


In [21]:
# Setting up the QA chain

os.environ["GROQ_API_KEY"] = "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"

# Initialize the LLM
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0,
)

# load vector store
vectorstore = FAISS.load_local(
    "faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

# create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

# Shoot queations

def ask_rag(question):
    print(f"\n{'='*60}")
    print(f"QUESTION: {question}")
    print('='*60)


    relevant_chunks = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in relevant_chunks])
    prompt = f"""You are a helpful research assistant analyzing ML research papers.
Use the context below to answer the question as best you can.
If the context contains partial information, use it to give the best possible answer.
Always mention which paper the information comes from.

Context:
{context}

Question: {question}

Answer:"""


    response = llm.invoke(prompt)
    answer = response.content

    print(f"\nANSWER:\n{answer}")

    print(f"\nSOURCES USED:")
    seen = set()
    for doc in relevant_chunks:
        source = doc.metadata.get('source', 'unknown').replace('papers/', '')
        page = doc.metadata.get('page', 0) + 1
        key = f"{source}_{page}"
        if key not in seen:
            seen.add(key)
            print(f"  📄 {source} — Page {page}")

    return answer, relevant_chunks

# Test it

ask_rag("What do BERT and GPT have in common?")
ask_rag("How does LoRA reduce training cost?")
ask_rag("Which models use the Transformer architecture?")


QUESTION: What do BERT and GPT have in common?

ANSWER:
According to the paper, BERT and GPT have the following in common:

1. Both are pre-training methods for natural language processing (NLP) tasks.
2. Both are trained on a large text corpus: BERT is trained on the BooksCorpus (800M words) and Wikipedia (2,500M words), while GPT is trained on the BooksCorpus (800M words).
3. Both use a Transformer-based architecture, although BERT uses a bidirectional Transformer and GPT uses a left-to-right Transformer.
4. Both are fine-tuning approaches, meaning they are trained on a specific task after pre-training on a large corpus.

The paper also mentions that many of the design decisions in BERT were intentionally made to make it as close to GPT as possible so that the two methods could be minimally compared.

SOURCES USED:
  📄 bert.pdf — Page 14
  📄 bert.pdf — Page 13
  📄 bert.pdf — Page 3
  📄 gpt3.pdf — Page 20
  📄 gpt3.pdf — Page 18
  📄 bert.pdf — Page 6

QUESTION: How does LoRA reduce tr

('The following models use the Transformer architecture:\n\n1. Transformer (Vaswani et al., 2017) - a sequence-to-sequence architecture that makes heavy use of self-attention.\n2. BERT (Devlin et al., 2019b) - a large Transformer language model.\n3. GPT-2 (Radford et al., b) - a large Transformer language model.\n\nAdditionally, the paper mentions that the BERT model is based on the original Transformer implementation described in Vaswani et al. (2017) and released in the tensor2tensor library.\n\nReferences:\n- Vaswani et al. (2017)\n- Devlin et al. (2019b)\n- Radford et al. (b)',
 [Document(id='7b4fe47e-b20f-43c6-a099-7161f022a1d9', metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-10-19T00:57:13+00:00', 'author': '', 'keywords': '', 'moddate': '2021-10-19T00:57:13+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '